In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from copy import deepcopy

import nltk
from nltk.tokenize import word_tokenize
# nltk.download('punkt_tab') # A faire la première fois

import seaborn as sns
import matplotlib.pyplot as plt

import unicodedata
import string

from visuEmbedding import components_to_fig_3D, components_to_fig_3D_animation
from modelSGNS import OnlyOneEmb, SGNS_OneEmbWeighted, SkipGramModel, SGNS_Weighted
from data.pipData import pipe_data, prepare_data, prepare_data_with_intonation, separate_text_intonation
import tool

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import skew

from collections import Counter

from data.pipData import separate_text_intonation
from dataSet import W2V_weighted_DataSet, W2V_weighted_DataSet_v2, dataset_weighted, SGNS_store_DataSet

import random

from typing import Callable, List, Type

[nltk_data] Downloading package punkt_tab to /home/pe/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/pe/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


# Fct

In [2]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # if you use multi-GPU
    # For absolute reproducibility (may slow down training slightly):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
def createEmbeddingWithSeed(seed:list[int],
                            dataset:Dataset,
                            model:nn.Module,
                            re_init_fct:Callable,
                            get_embedding:Callable,
                            optimizer_cls:type[torch.optim.Optimizer],
                            lr:float, nb_epoch:int,
                            path_to_save:str,
                            words:list[str],
                            device: str = "cpu",
                            batch_size: int = 16,
                            num_workers: int = 0
                        ):
    model.to(device)
    loss_by_seed: List[List[float]] = []
    
    for s in seed:
        set_all_seeds(s)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
        
        model = re_init_fct(model)
        optimizer = optimizer_cls(model.parameters(), lr=lr)
        
        loss_by_epoch:list[float] = []
        for epoch in range(nb_epoch):
            loss_in_epoch:list[float] = []
            for sentence_nb, data in enumerate(loader):
                if isinstance(data, (list, tuple)):
                    data = [d.to(device) for d in data]
                else:
                    data = data.to(device)
                    
                optimizer.zero_grad()
                loss:torch.Tensor = model(data)
                loss.backward()
                optimizer.step()
                loss_in_epoch.append(loss.detach().cpu().item())
            loss_by_epoch.append(np.mean(loss_in_epoch))
        
        vectors = get_embedding(model)
        model_name = type(model).__name__
        np.savez(
            f'{path_to_save}/seed_{s}_{model_name}_.npz', 
            vectors=vectors,
            words=words
        )
        loss_by_seed.append(loss_by_epoch)
        print(f"For the seed {s} we have loss {loss_by_epoch}")
        
    
    return loss_by_seed

# Data

## GNG

In [10]:
data = prepare_data_with_intonation(
    file_path="./data/GoodNightGorilla_Intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={# specific to corpus 
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",},
    stop_words=["s", "n't"],
    break_line=False
)
texts, intonations = separate_text_intonation(data)

## I went Walking

In [3]:
data = prepare_data_with_intonation(
    file_path="data/IwentWalking_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)

texts, intonations = separate_text_intonation(data)


# TRDP

In [14]:
data = prepare_data_with_intonation(
    file_path="data/TheRainyDayParade_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)

texts, intonations = separate_text_intonation(data)


## GNG + I went Walking

In [4]:
data = prepare_data_with_intonation(
    file_path="./data/GoodNightGorilla_Intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={# specific to corpus 
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",},
    stop_words=["s", "n't"],
    break_line=False
)

texts, intonations = separate_text_intonation(data)

print(texts)

data = prepare_data_with_intonation(
    file_path="data/IwentWalking_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)

texts2, intonations2 = separate_text_intonation(data)
intonations.extend(intonations2)
texts.extend(texts2)
print(texts)

[['look', 'there', 'is', 'the', 'zookeeper', 'he', 'has', 'a', 'big', 'flashlight', 'to', 'see', 'in', 'the', 'dark', 'click', 'what', 'is', 'he', 'saying', 'to', 'the', 'animal', 'he', 'says', 'good', 'night', 'gorilla', 'can', 'you', 'say', 'good', 'night'], ['oh', 'my', 'goodness', 'look', 'closer', 'is', 'the', 'gorilla', 'going', 'to', 'sleep', 'no', 'he', 'is', 'reaching', 'out', 'and', 'taking', 'the', 'keys', 'that', 'sneaky', 'gorilla', 'is', 'stealing', 'the', 'keys', 'right', 'off', 'the', 'zookeeper', 'belt', 'jingle', 'jangle'], ['who', 'sees', 'him', 'doing', 'it', 'it', 'the', 'little', 'mouse', 'squeak', 'squeak', 'the', 'mouse', 'is', 'watching', 'everything'], ['look', 'at', 'the', 'gorilla', 'room', 'he', 'has', 'a', 'bicycle', 'in', 'there', 'and', 'a', 'big', 'tire', 'to', 'swing', 'on', 'but', 'he', 'doesnot', 'want', 'to', 'stay', 'inside', 'does', 'he', 'he', 'wants', 'to', 'follow', 'the', 'zookeeper'], ['and', 'what', 'is', 'that', 'pink', 'thing', 'floating',

## TS

In [5]:
data = prepare_data_with_intonation(
    file_path="data/TheSnowman_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)

texts, intonations = separate_text_intonation(data)

## TS + GNG + IWW

In [6]:
data = prepare_data_with_intonation(
    file_path="./data/GoodNightGorilla_Intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={# specific to corpus 
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",},
    stop_words=["s", "n't"],
    break_line=False
)

texts, intonations = separate_text_intonation(data)

print(texts)

data = prepare_data_with_intonation(
    file_path="data/IwentWalking_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)

texts2, intonations2 = separate_text_intonation(data)
intonations.extend(intonations2)
texts.extend(texts2)

data = prepare_data_with_intonation(
    file_path="data/TheSnowman_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)

texts2, intonations2 = separate_text_intonation(data)
intonations.extend(intonations2)
texts.extend(texts2)
print(texts)

[['look', 'there', 'is', 'the', 'zookeeper', 'he', 'has', 'a', 'big', 'flashlight', 'to', 'see', 'in', 'the', 'dark', 'click', 'what', 'is', 'he', 'saying', 'to', 'the', 'animal', 'he', 'says', 'good', 'night', 'gorilla', 'can', 'you', 'say', 'good', 'night'], ['oh', 'my', 'goodness', 'look', 'closer', 'is', 'the', 'gorilla', 'going', 'to', 'sleep', 'no', 'he', 'is', 'reaching', 'out', 'and', 'taking', 'the', 'keys', 'that', 'sneaky', 'gorilla', 'is', 'stealing', 'the', 'keys', 'right', 'off', 'the', 'zookeeper', 'belt', 'jingle', 'jangle'], ['who', 'sees', 'him', 'doing', 'it', 'it', 'the', 'little', 'mouse', 'squeak', 'squeak', 'the', 'mouse', 'is', 'watching', 'everything'], ['look', 'at', 'the', 'gorilla', 'room', 'he', 'has', 'a', 'bicycle', 'in', 'there', 'and', 'a', 'big', 'tire', 'to', 'swing', 'on', 'but', 'he', 'doesnot', 'want', 'to', 'stay', 'inside', 'does', 'he', 'he', 'wants', 'to', 'follow', 'the', 'zookeeper'], ['and', 'what', 'is', 'that', 'pink', 'thing', 'floating',

## TGTR

In [7]:
data = prepare_data_with_intonation(
    file_path="data/TheGreatToyRescue_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)

texts, intonations = separate_text_intonation(data)

## TSGE

In [8]:
data = prepare_data_with_intonation(
    file_path="data/TheShadowGreatEscape_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)

texts, intonations = separate_text_intonation(data)

# GNG + IWW + TSGE

In [22]:
data = prepare_data_with_intonation(
    file_path="./data/GoodNightGorilla_Intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={# specific to corpus 
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",},
    stop_words=["s", "n't"],
    break_line=False
)

texts, intonations = separate_text_intonation(data)

print(texts)

data = prepare_data_with_intonation(
    file_path="data/IwentWalking_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)

texts2, intonations2 = separate_text_intonation(data)
intonations.extend(intonations2)
texts.extend(texts2)

data = prepare_data_with_intonation(
    file_path="data/TheShadowGreatEscape_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)

texts2, intonations2 = separate_text_intonation(data)
intonations.extend(intonations2)
texts.extend(texts2)
print(texts)

[['look', 'there', 'is', 'the', 'zookeeper', 'he', 'has', 'a', 'big', 'flashlight', 'to', 'see', 'in', 'the', 'dark', 'click', 'what', 'is', 'he', 'saying', 'to', 'the', 'animal', 'he', 'says', 'good', 'night', 'gorilla', 'can', 'you', 'say', 'good', 'night'], ['oh', 'my', 'goodness', 'look', 'closer', 'is', 'the', 'gorilla', 'going', 'to', 'sleep', 'no', 'he', 'is', 'reaching', 'out', 'and', 'taking', 'the', 'keys', 'that', 'sneaky', 'gorilla', 'is', 'stealing', 'the', 'keys', 'right', 'off', 'the', 'zookeeper', 'belt', 'jingle', 'jangle'], ['who', 'sees', 'him', 'doing', 'it', 'it', 'the', 'little', 'mouse', 'squeak', 'squeak', 'the', 'mouse', 'is', 'watching', 'everything'], ['look', 'at', 'the', 'gorilla', 'room', 'he', 'has', 'a', 'bicycle', 'in', 'there', 'and', 'a', 'big', 'tire', 'to', 'swing', 'on', 'but', 'he', 'doesnot', 'want', 'to', 'stay', 'inside', 'does', 'he', 'he', 'wants', 'to', 'follow', 'the', 'zookeeper'], ['and', 'what', 'is', 'that', 'pink', 'thing', 'floating',

# GNG IWW TSGE TRDP

# Norm
norm01 : range_norm = 1.9 and center_norm = 1.

norm02 : range_norm = 1.75 and center_norm = 1.

norm03 : range_norm = 1.5 and center_norm = 1.


In [20]:
range_norm = 1.9
center_norm = 1.
intonations_normalize = tool.normalize_range_center(intonations, range_normalize=range_norm, center=center_norm)

dataset:dataset_weighted = dataset_weighted(sentences=texts,
                                intonations=intonations_normalize, nb_neg=10, window_size=6)

intonations_binaire = tool.make_binary(intonations)
print(dataset.vocab)


['look', 'at', 'the', 'cover', 'of', 'this', 'big', 'book', 'what', 'do', 'you', 'see', 'right', 'here', 'itis', 'a', 'little', 'girl', 'her', 'smile', 'she', 'looks', 'so', 'happy', 'and', 'is', 'wearing', 'has', 'bright', 'yellow', 'raincoat', 'on', 'it', 'very', 'shiny', 'head', 'matching', 'hat', 'why', 'all', 'those', 'clothes', 'up', 'sky', 'clouds', 'they', 'gray', 'puffy', 'drip', 'drop', 'raining', 'but', 'doesnot', 'sad', 'loves', 'rain', 'wants', 'to', 'go', 'outside', 'play', 'are', 'ready', 'rainy', 'day', 'adventure', 'with', 'let', 'open', 'put', 'your', 'hand', 'corner', 'one', 'two', 'three', 'flip', 'oh', 'my', 'goodness', 'messy', 'room', 'getting', 'coat', 'feet', 'shoes', 'no', 'silly', 'polkadot', 'socks', 'ca', 'in', 'would', 'get', 'wet', 'cold', 'brrr', 'needs', 'boots', 'can', 'find', 'picture', 'under', 'bed', 'there', 'thatis', 'teddy', 'bear', 'chair', 'kitty', 'cat', 'toy', 'wait', 'over', 'by', 'door', 'yes', 'found', 'them', 'putting', 'now', 'push', 'op

# My method

In [21]:
model = SGNS_OneEmbWeighted(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

def re_init_model_w2v_weighted(model:SGNS_OneEmbWeighted):
    with torch.no_grad():
        model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
    return model

def get_embedding_w2v_weighted(model:SGNS_OneEmbWeighted):
    return model.word_emb.weight.detach().cpu().numpy()

createEmbeddingWithSeed(seed = range(0, 10),
                        dataset=dataset,
                        model=model,
                        re_init_fct=re_init_model_w2v_weighted,
                        get_embedding=get_embedding_w2v_weighted,
                        optimizer_cls=torch.optim.Adam,
                        lr=0.003, nb_epoch=15,
                        path_to_save="embedding/TRDP/norm01",
                        words=list(dataset.encoder.keys()),
                        device="cuda",
                        batch_size= 16,
                        num_workers= 0)

For the seed 0 we have loss [np.float64(5.370828704679125), np.float64(5.303890389905731), np.float64(5.262970288003088), np.float64(5.247513692175745), np.float64(5.230569224396319), np.float64(5.2145861009783605), np.float64(5.215249430181531), np.float64(5.201724657993033), np.float64(5.1956383054408075), np.float64(5.199717604904278), np.float64(5.200466133422232), np.float64(5.199292116623286), np.float64(5.183646870885392), np.float64(5.18477336818375), np.float64(5.193247967385794)]
For the seed 1 we have loss [np.float64(5.373253124815202), np.float64(5.3088187248684235), np.float64(5.265327668480364), np.float64(5.244822952195659), np.float64(5.233804221566217), np.float64(5.226209952640921), np.float64(5.2256419663500235), np.float64(5.2034977903546755), np.float64(5.2076163874266115), np.float64(5.2029144570372585), np.float64(5.197014538780439), np.float64(5.198534750648054), np.float64(5.198770309010765), np.float64(5.1909292292691696), np.float64(5.191002870767462)]
For t

[[np.float64(5.370828704679125),
  np.float64(5.303890389905731),
  np.float64(5.262970288003088),
  np.float64(5.247513692175745),
  np.float64(5.230569224396319),
  np.float64(5.2145861009783605),
  np.float64(5.215249430181531),
  np.float64(5.201724657993033),
  np.float64(5.1956383054408075),
  np.float64(5.199717604904278),
  np.float64(5.200466133422232),
  np.float64(5.199292116623286),
  np.float64(5.183646870885392),
  np.float64(5.18477336818375),
  np.float64(5.193247967385794)],
 [np.float64(5.373253124815202),
  np.float64(5.3088187248684235),
  np.float64(5.265327668480364),
  np.float64(5.244822952195659),
  np.float64(5.233804221566217),
  np.float64(5.226209952640921),
  np.float64(5.2256419663500235),
  np.float64(5.2034977903546755),
  np.float64(5.2076163874266115),
  np.float64(5.2029144570372585),
  np.float64(5.197014538780439),
  np.float64(5.198534750648054),
  np.float64(5.198770309010765),
  np.float64(5.1909292292691696),
  np.float64(5.191002870767462)],
 

# Two embedding weighted

In [22]:
model = SGNS_Weighted(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

def re_init_model_w2v_weighted(model:SGNS_Weighted):
    with torch.no_grad():
        model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
        model.con_emb.weight.data.uniform_(-model.init_range, model.init_range)
    return model

def get_embedding_w2v_weighted(model:SGNS_Weighted):
    return model.word_emb.weight.detach().cpu().numpy()

createEmbeddingWithSeed(seed = range(0, 10),
                        dataset=dataset,
                        model=model,
                        re_init_fct=re_init_model_w2v_weighted,
                        get_embedding=get_embedding_w2v_weighted,
                        optimizer_cls=torch.optim.Adam,
                        lr=0.003, nb_epoch=15,
                        path_to_save="embedding/TRDP/norm01",
                        words=list(dataset.encoder.keys()),
                        device="cuda",
                        batch_size= 16,
                        num_workers= 0)

For the seed 0 we have loss [np.float64(3.948666036370965), np.float64(2.4361106247637365), np.float64(2.3033402590370953), np.float64(2.2529277449854335), np.float64(2.1890814740699747), np.float64(2.1006246741472947), np.float64(2.006545942432987), np.float64(1.909439510599041), np.float64(1.8222073230272378), np.float64(1.7558729969600217), np.float64(1.6988801399248379), np.float64(1.6535100490056451), np.float64(1.6030651955223858), np.float64(1.5693293153352441), np.float64(1.5500122477301725)]
For the seed 1 we have loss [np.float64(3.908443830010049), np.float64(2.433969503649194), np.float64(2.3078902815449704), np.float64(2.259624980863925), np.float64(2.196201408990181), np.float64(2.1133226299802086), np.float64(2.015027042775741), np.float64(1.9204545603795369), np.float64(1.840835844147673), np.float64(1.7699106145695518), np.float64(1.7052143318565998), np.float64(1.6559257291166645), np.float64(1.6097434716392434), np.float64(1.5835515303040713), np.float64(1.5461568800

[[np.float64(3.948666036370965),
  np.float64(2.4361106247637365),
  np.float64(2.3033402590370953),
  np.float64(2.2529277449854335),
  np.float64(2.1890814740699747),
  np.float64(2.1006246741472947),
  np.float64(2.006545942432987),
  np.float64(1.909439510599041),
  np.float64(1.8222073230272378),
  np.float64(1.7558729969600217),
  np.float64(1.6988801399248379),
  np.float64(1.6535100490056451),
  np.float64(1.6030651955223858),
  np.float64(1.5693293153352441),
  np.float64(1.5500122477301725)],
 [np.float64(3.908443830010049),
  np.float64(2.433969503649194),
  np.float64(2.3078902815449704),
  np.float64(2.259624980863925),
  np.float64(2.196201408990181),
  np.float64(2.1133226299802086),
  np.float64(2.015027042775741),
  np.float64(1.9204545603795369),
  np.float64(1.840835844147673),
  np.float64(1.7699106145695518),
  np.float64(1.7052143318565998),
  np.float64(1.6559257291166645),
  np.float64(1.6097434716392434),
  np.float64(1.5835515303040713),
  np.float64(1.5461568

# OnlyOneEmb

In [18]:
dataset = SGNS_store_DataSet(sentences=texts, nb_neg=10, power=0.75, subsample_thresh=0, window_size=6)

model = OnlyOneEmb(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

def re_init_model_w2v_one_emb(model:OnlyOneEmb):
    with torch.no_grad():
        model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
    return model

def get_embedding_w2v_one_emb(model:OnlyOneEmb):
    return model.word_emb.weight.detach().cpu().numpy()

createEmbeddingWithSeed(seed = range(0, 10), 
                        dataset=dataset,
                        model=model,
                        re_init_fct=re_init_model_w2v_one_emb,
                        get_embedding=get_embedding_w2v_one_emb,
                        optimizer_cls=torch.optim.Adam,
                        lr=0.003, nb_epoch=15,
                        path_to_save="embedding/TRDP/noNorm",
                        words=list(dataset.encoder.keys()),
                        device="cuda",
                        batch_size= 16,
                        num_workers= 0)

For the seed 0 we have loss [np.float64(7.621051074687457), np.float64(7.609340867105776), np.float64(7.597686419145019), np.float64(7.590594990488963), np.float64(7.580629101625476), np.float64(7.573019357430919), np.float64(7.573161993039638), np.float64(7.570112690390044), np.float64(7.568764780145214), np.float64(7.565682668647199), np.float64(7.567908060566821), np.float64(7.563271628987644), np.float64(7.56659273526988), np.float64(7.563479674523835), np.float64(7.564974013136268)]
For the seed 1 we have loss [np.float64(7.619024571611046), np.float64(7.604424623094489), np.float64(7.5978391625400485), np.float64(7.587872802646621), np.float64(7.580055236816406), np.float64(7.573953025234569), np.float64(7.577722553309956), np.float64(7.57087650157118), np.float64(7.564268861313796), np.float64(7.572424128510471), np.float64(7.563560987196369), np.float64(7.564841977636934), np.float64(7.569093105434887), np.float64(7.571933274017458), np.float64(7.558626461416201)]
For the seed 

[[np.float64(7.621051074687457),
  np.float64(7.609340867105776),
  np.float64(7.597686419145019),
  np.float64(7.590594990488963),
  np.float64(7.580629101625476),
  np.float64(7.573019357430919),
  np.float64(7.573161993039638),
  np.float64(7.570112690390044),
  np.float64(7.568764780145214),
  np.float64(7.565682668647199),
  np.float64(7.567908060566821),
  np.float64(7.563271628987644),
  np.float64(7.56659273526988),
  np.float64(7.563479674523835),
  np.float64(7.564974013136268)],
 [np.float64(7.619024571611046),
  np.float64(7.604424623094489),
  np.float64(7.5978391625400485),
  np.float64(7.587872802646621),
  np.float64(7.580055236816406),
  np.float64(7.573953025234569),
  np.float64(7.577722553309956),
  np.float64(7.57087650157118),
  np.float64(7.564268861313796),
  np.float64(7.572424128510471),
  np.float64(7.563560987196369),
  np.float64(7.564841977636934),
  np.float64(7.569093105434887),
  np.float64(7.571933274017458),
  np.float64(7.558626461416201)],
 [np.floa

# SkipGramModel

In [17]:
dataset = SGNS_store_DataSet(sentences=texts, nb_neg=10, power=0.75, subsample_thresh=0, window_size=6)

model = SkipGramModel(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

def re_init_model_w2v_two_emb(model:SkipGramModel):
    with torch.no_grad():
        model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
        model.con_emb.weight.data.uniform_(-model.init_range, model.init_range)
    return model

def get_embedding_w2v_two_emb(model:SkipGramModel):
    return model.word_emb.weight.detach().cpu().numpy()

createEmbeddingWithSeed(seed = range(0, 10),
                        dataset=dataset,
                        model=model,
                        re_init_fct=re_init_model_w2v_two_emb,
                        get_embedding=get_embedding_w2v_two_emb,
                        optimizer_cls=torch.optim.Adam,
                        lr=0.003, nb_epoch=25,
                        path_to_save="embedding/TRDP/noNorm",
                        words=list(dataset.encoder.keys()),
                        device="cuda",
                        batch_size= 16,
                        num_workers= 0)

For the seed 0 we have loss [np.float64(4.972547989898186), np.float64(3.38692782696271), np.float64(3.287479792299387), np.float64(3.255503119570638), np.float64(3.192455739871736), np.float64(3.1011505359241864), np.float64(2.9941123699787022), np.float64(2.89372315651993), np.float64(2.8026100285804967), np.float64(2.7279466230911233), np.float64(2.6723470181347713), np.float64(2.6257569296272907), np.float64(2.5933397414719783), np.float64(2.560590020374613), np.float64(2.5438063560867827), np.float64(2.5117515449756214), np.float64(2.4958158171709237), np.float64(2.4835393264264636), np.float64(2.479329611193666), np.float64(2.4659775737496608), np.float64(2.4656219505005503), np.float64(2.451703626183599), np.float64(2.439994213706908), np.float64(2.4311943989806633), np.float64(2.4355955552990287)]
For the seed 1 we have loss [np.float64(4.895785885024941), np.float64(3.3895407710249597), np.float64(3.289811502451503), np.float64(3.256981179259304), np.float64(3.1933039338082194

[[np.float64(4.972547989898186),
  np.float64(3.38692782696271),
  np.float64(3.287479792299387),
  np.float64(3.255503119570638),
  np.float64(3.192455739871736),
  np.float64(3.1011505359241864),
  np.float64(2.9941123699787022),
  np.float64(2.89372315651993),
  np.float64(2.8026100285804967),
  np.float64(2.7279466230911233),
  np.float64(2.6723470181347713),
  np.float64(2.6257569296272907),
  np.float64(2.5933397414719783),
  np.float64(2.560590020374613),
  np.float64(2.5438063560867827),
  np.float64(2.5117515449756214),
  np.float64(2.4958158171709237),
  np.float64(2.4835393264264636),
  np.float64(2.479329611193666),
  np.float64(2.4659775737496608),
  np.float64(2.4656219505005503),
  np.float64(2.451703626183599),
  np.float64(2.439994213706908),
  np.float64(2.4311943989806633),
  np.float64(2.4355955552990287)],
 [np.float64(4.895785885024941),
  np.float64(3.3895407710249597),
  np.float64(3.289811502451503),
  np.float64(3.256981179259304),
  np.float64(3.1933039338082